In [1]:
import numpy as np
import pandas as pd

# Load country datasets
df_in = pd.read_csv("/content/INvideos.csv")
df_us = pd.read_csv("/content/USvideos.csv")
df_gb = pd.read_csv("/content/GBvideos.csv")

categories = pd.read_csv("/content/categories.csv")

# Add country identifiers
df_in["country"] = "India"
df_us["country"] = "USA"
df_gb["country"] = "Great Britain"

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [2]:
# Combine country datasets
df = pd.concat([df_in, df_us, df_gb], ignore_index=True)

print("Combined dataset shape:", df.shape)
print("\nCountry distribution:")
print(df["country"].value_counts())

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary:")
display(df.describe().T)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

Combined dataset shape: (46230, 17)

Country distribution:
country
USA              16080
India            15075
Great Britain    15075
Name: count, dtype: int64

Data types:
video_id                   object
trending_date              object
title                      object
channel_title              object
category_id                 int64
publish_time               object
tags                       object
views                       int64
likes                     float64
dislikes                    int64
comment_count             float64
thumbnail_link             object
comments_disabled            bool
ratings_disabled             bool
video_error_or_removed       bool
description                object
country                    object
dtype: object

Statistical summary:


,count,mean,std,min,25%,50%,75%,max
category_id,46230.0,1.864534e+01,7.646745e+00,1.0,10.00,22.0,24.00,28.0
views,46230.0,1.178603e+06,5.916188e+06,1066.0,129463.25,318277.5,824050.50,360760164.0
likes,46218.0,4.580005e+04,2.969554e+05,0.0,3656.00,10201.0,28987.75,27537822.0
dislikes,46230.0,2.325984e+03,1.051896e+04,0.0,213.00,594.0,1666.00,618175.0
comment_count,46194.0,3.449773e+03,1.366515e+04,0.0,404.00,1074.0,2779.00,778390.0



Missing values:
video_id                     0
trending_date                0
title                        0
channel_title                0
category_id                  0
publish_time                 0
tags                         0
views                        0
likes                       12
dislikes                     0
comment_count               36
thumbnail_link               0
comments_disabled            0
ratings_disabled             0
video_error_or_removed       0
description               1624
country                      0
dtype: int64

Duplicate records: 230


In [3]:
views = df["views"].to_numpy()
likes = df["likes"].to_numpy()
comments = df["comment_count"].to_numpy()

print("Views array shape:", views.shape)
print("Views dimensions:", views.ndim)
print("Views data type:", views.dtype)

print("\nTotal views:", np.nansum(views))
print("Average views:", np.nanmean(views))
print("Maximum views:", np.nanmax(views))
print("Minimum views:", np.nanmin(views))

Views array shape: (46230,)
Views dimensions: 1
Views data type: int64

Total views: 54486825046
Average views: 1178603.1807484317
Maximum views: 360760164
Minimum views: 1066


In [4]:
# Calculate total engagement
df["total_engagement"] = df[
    ["likes", "comment_count"]
].sum(axis=1, min_count=2)

# Calculate engagement rate
df["engagement_rate"] = np.where(
    df["views"] > 0,
    (df["total_engagement"] / df["views"]) * 100,
    np.nan
)

print(df[[
    "video_id", "views", "likes",
    "comment_count", "total_engagement",
    "engagement_rate"
]].head())

      video_id   views    likes  comment_count  total_engagement  \
0  UcYiQY22yfG  217915  14918.0          662.0           15580.0   
1  EutcTEIciae  115014   2739.0          209.0            2948.0   
2  rvuZ5CsBpg8   51028   1786.0          365.0            2151.0   
3  2yJGS3uxgMP  143370   5224.0          571.0            5795.0   
4  W5pdmLZu7GK  390220  15079.0         1942.0           17021.0   

   engagement_rate  
0         7.149577  
1         2.563166  
2         4.215333  
3         4.041989  
4         4.361898  


In [5]:
# Min-max normalisation
minimum = df["views"].min()
maximum = df["views"].max()

df["normalized_views"] = (
    (df["views"] - minimum) / (maximum - minimum)
)

print(df[["views", "normalized_views"]].head())

# NumPy broadcasting
sample_array = np.array([
    [100, 200, 300],
    [400, 500, 600],
    [700, 800, 900]
])

adjustment = np.array([10, 20, 30])

broadcast_result = sample_array + adjustment

print("\nBroadcasting result:")
print(broadcast_result)

    views  normalized_views
0  217915          0.000601
1  115014          0.000316
2   51028          0.000138
3  143370          0.000394
4  390220          0.001079

Broadcasting result:
[[110 220 330]
 [410 520 630]
 [710 820 930]]


In [6]:
# Create a clean working copy
df_clean = df.drop_duplicates().copy()

# Fill missing descriptions with empty strings
df_clean["description"] = df_clean["description"].fillna("")

# Identify missing engagement values
print("Missing likes:", df_clean["likes"].isnull().sum())
print("Missing comments:", df_clean["comment_count"].isnull().sum())

# Filter videos with more than one million views
viral_videos = df_clean[df_clean["views"] >= 1000000]

# Filter Indian videos
india_videos = df_clean[df_clean["country"] == "India"]

print("\nViral videos:", len(viral_videos))
print("Indian videos:", len(india_videos))
print("Cleaned dataset shape:", df_clean.shape)

Missing likes: 12
Missing comments: 36

Viral videos: 9686
Indian videos: 15000
Cleaned dataset shape: (46000, 20)


In [7]:
print("Category columns:", categories.columns.tolist())

# Match category reference using category_id
category_df = categories.drop_duplicates(
    subset=["category_id"]
).copy()

df_merged = pd.merge(
    df_clean,
    category_df,
    on="category_id",
    how="left",
    suffixes=("", "_category")
)

print("Merged dataset shape:", df_merged.shape)
display(df_merged.head())

Category columns: ['category_id', 'category_name']
Merged dataset shape: (46000, 21)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,total_engagement,engagement_rate,normalized_views,category_name
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,...,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,India,15580.0,7.149577,0.000601,Music
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,...,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,India,2948.0,2.563166,0.000316,Film & Animation
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,...,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,India,2151.0,4.215333,0.000138,Entertainment
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,...,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,India,5795.0,4.041989,0.000394,Comedy
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,...,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,India,17021.0,4.361898,0.001079,Entertainment


In [8]:
# Country-wise analysis
country_analysis = df_clean.groupby("country").agg(
    total_videos=("video_id", "count"),
    average_views=("views", "mean"),
    maximum_views=("views", "max"),
    average_likes=("likes", "mean")
).reset_index()

display(country_analysis)

# Category-wise analysis
category_analysis = df_clean.groupby("category_id").agg(
    total_videos=("video_id", "count"),
    average_views=("views", "mean"),
    average_engagement=("engagement_rate", "mean")
).reset_index()

display(category_analysis.head(10))

,country,total_videos,average_views,maximum_views,average_likes
0,Great Britain,15000,1.584041e+06,360760164,65266.896439
1,India,15000,9.521008e+05,123208090,33144.305948
2,USA,16000,1.013904e+06,129229245,39560.292323


,category_id,total_videos,average_views,average_engagement
0,1,2297,5.779740e+05,3.533814
1,2,1007,2.796927e+05,3.381299
2,10,10310,3.190950e+06,4.423679
3,15,913,3.497486e+05,4.149922
4,17,1921,3.749782e+05,2.993939
5,19,709,2.291177e+05,3.458641
6,20,2057,5.096649e+05,4.678649
7,22,3804,5.512341e+05,4.326598
8,23,3258,6.034757e+05,5.004631
9,24,12270,9.010256e+05,3.581130


In [9]:
# Convert dates
df_clean["publish_time"] = pd.to_datetime(
    df_clean["publish_time"],
    errors="coerce",
    utc=True
)

df_clean["trending_date"] = pd.to_datetime(
    df_clean["trending_date"],
    format="%y.%d.%m",
    errors="coerce"
)

# Remove timezone from publication timestamp for subtraction
df_clean["publish_time"] = (
    df_clean["publish_time"].dt.tz_convert(None)
)

# Calculate time taken to trend
df_clean["days_to_trend"] = (
    df_clean["trending_date"].dt.normalize()
    - df_clean["publish_time"].dt.normalize()
).dt.days

print(df_clean[[
    "video_id", "publish_time",
    "trending_date", "days_to_trend"
]].head())

      video_id        publish_time trending_date  days_to_trend
0  UcYiQY22yfG 2017-11-14 05:57:11    2017-11-14              0
1  EutcTEIciae 2017-11-08 13:18:04    2017-11-14              6
2  rvuZ5CsBpg8 2017-11-14 18:16:59    2017-11-14              0
3  2yJGS3uxgMP 2017-11-11 20:04:47    2017-11-14              3
4  W5pdmLZu7GK 2017-11-14 05:23:55    2017-11-14              0


In [10]:
# Count distinct dates each video appeared in trending
trending_lifespan = df_clean.groupby(
    ["country", "video_id"]
).agg(
    trending_days=("trending_date", "nunique"),
    total_views=("views", "max"),
    average_engagement=("engagement_rate", "mean")
).reset_index()

print("Trending lifespan analysis:")
display(
    trending_lifespan.sort_values(
        "trending_days",
        ascending=False
    ).head(10)
)

Trending lifespan analysis:


,country,video_id,trending_days,total_views,average_engagement
1611,Great Britain,NqokCmZykzK,25,97513776,4.487590
1630,Great Britain,O6zdCRwWdf1,25,17542788,6.350387
1019,Great Britain,EZaG9_2Yi1u,25,38242672,4.854379
950,Great Britain,DYLzrqG-SrK,25,26603358,3.579111
1846,Great Britain,RdqSDDng84n,25,360760164,7.799198
1432,Great Britain,Kv1WBQp86cP,25,25378443,3.930728
3859,Great Britain,w1lZB-FX3Sn,25,10803119,2.961449
11006,USA,BsTMdCTYs7P,25,54982508,5.609326
2343,Great Britain,Z1aFIGyR5zv,25,8656350,5.187291
2687,Great Britain,dIbJkzpClRM,25,60754408,3.113217


In [11]:
df_clean.to_csv(
    "TrendPulse_Part2_Cleaned.csv",
    index=False
)

df_merged.to_csv(
    "TrendPulse_Part2_Merged.csv",
    index=False
)

country_analysis.to_csv(
    "Country_Analysis.csv",
    index=False
)

category_analysis.to_csv(
    "Category_Analysis.csv",
    index=False
)

trending_lifespan.to_csv(
    "Trending_Lifespan.csv",
    index=False
)

print("Part 2 outputs saved successfully!")

Part 2 outputs saved successfully!


In [12]:
print("========== PART 2 VERIFICATION ==========")

print("Total records:", len(df_clean))
print("NumPy arrays:", views.shape)
print("Engagement rate:", "engagement_rate" in df_clean.columns)
print("Normalisation:", "normalized_views" in df_clean.columns)
print("Data cleaning:", df_clean.duplicated().sum() == 0)
print("Merge completed:", len(df_merged) > 0)
print("GroupBy completed:", len(country_analysis) > 0)
print("Days-to-trend:", "days_to_trend" in df_clean.columns)
print("Trending lifespan:", len(trending_lifespan) > 0)

print("\nPART 2 PROCESSING FINISHED!")

========== PART 2 VERIFICATION ==========
Total records: 46000
NumPy arrays: (46230,)
Engagement rate: True
Normalisation: True
Data cleaning: True
Merge completed: True
GroupBy completed: True
Days-to-trend: True
Trending lifespan: True

PART 2 PROCESSING FINISHED!
